At each node, the tree asks:
> Which feature and threshold splits the data into purer groups?

For binary classification, a common impurity is Gini impurity:
$$G = 1 - \sum^C_{c=1} p_c^2$$

- $p_c$ is the fraction of class $c$ in that node
- if a node is perfectly pure, say all class 1, then $p_1 = 1$, so $G=0$
- if classes are mixed, then $G=0.5$

When splitting a node into left/right children, we want the weighted impurity after split to be as small as possible:
$$G_{split} = \frac{nL}{n}G_L + \frac{nR}{n}G_R$$

Choose the features $j$ and threshold $t$ that minimize this.

so training is:
1. try many candidate splits
2. pick the best one
3. recurse on left/right subsets
4. stop when max depth reached, node too small, or pure enough

In [1]:
import numpy as np

class DecisionTree:
    def __init__(self, max_depth=None):
        self.max_depth = max_depth
        
    def fit(self, X, y):
        self.n_classes_ = len(np.unique(y))
        self.n_features_ = X.shape[1]
        self.tree_ = self._grow_tree(X, y)
        
    def predict(self, X):
        return [self._predict(inputs) for inputs in X]
        
    def _gini(self, y):
        _, counts = np.unique(y, return_counts=True)
        impurity = 1 - np.sum([(count / len(y)) ** 2 for count in counts])
        return impurity
        
    def _best_split(self, X, y):
        m = y.size
        if m <= 1:
            return None, None
        
        num_parent = [np.sum(y == c) for c in range(self.n_classes_)]
        best_gini = 1.0 - sum((n / m) ** 2 for n in num_parent)
        best_idx, best_thr = None, None
        
        for idx in range(self.n_features_):
            thresholds, classes = zip(*sorted(zip(X[:, idx], y)))
            num_left = [0] * self.n_classes_
            num_right = num_parent.copy()
            for i in range(1, m):
                c = classes[i - 1]
                num_left[c] += 1
                num_right[c] -= 1
                gini_left = 1.0 - sum(
                    (num_left[x] / i) ** 2 for x in range(self.n_classes_)
                )
                gini_right = 1.0 - sum(
                    (num_right[x] / (m - i)) ** 2 for x in range(self.n_classes_)
                )
                gini = (i * gini_left + (m - i) * gini_right) / m
                if thresholds[i] == thresholds[i - 1]:
                    continue
                if gini < best_gini:
                    best_gini = gini
                    best_idx = idx
                    best_thr = (thresholds[i] + thresholds[i - 1]) / 2
        
        return best_idx, best_thr
        
    def _grow_tree(self, X, y, depth=0):
        num_samples_per_class = [np.sum(y == i) for i in range(self.n_classes_)]
        predicted_class = np.argmax(num_samples_per_class)
        node = Node(predicted_class=predicted_class)
        if depth < self.max_depth:
            idx, thr = self._best_split(X, y)
            if idx is not None:
                indices_left = X[:, idx] < thr
                X_left, y_left = X[indices_left], y[indices_left]
                X_right, y_right = X[~indices_left], y[~indices_left]
                node.feature_index = idx
                node.threshold = thr
                node.left = self._grow_tree(X_left, y_left, depth + 1)
                node.right = self._grow_tree(X_right, y_right, depth + 1)
        return node
        
    def _predict(self, inputs):
        node = self.tree_
        while node.left:
            if inputs[node.feature_index] < node.threshold:
                node = node.left
            else:
                node = node.right
        return node.predicted_class
    
class Node:
    def __init__(self, *, predicted_class):
        self.predicted_class = predicted_class
        self.feature_index = 0
        self.threshold = 0.0 
        self.left = None
        self.right = None

    def is_leaf_node(self):
        return self.left is None and self.right is None

### Booster Wrapper

In [3]:
class SimpleXGBoostClassifier:
    def __init__(
        self,
        n_estimators=20,
        learning_rate=0.1,
        max_depth=3,
        min_samples_split=2,
        reg_lambda=1.0,
        gamma=0.0,
    ):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.reg_lambda = reg_lambda
        self.gamma = gamma
        self.trees = []
        self.base_score = 0.0

    def _sigmoid(self, x):
        return torch.sigmoid(x)

    def fit(self, X, y):
        """
        X: [N, D] float tensor
        y: [N] float tensor with values 0 or 1
        """
        eps = 1e-8
        p = y.mean().clamp(eps, 1 - eps)
        self.base_score = torch.log(p / (1 - p)).item()

        y_hat = torch.full_like(y, fill_value=self.base_score)

        self.trees = []

        for t in range(self.n_estimators):
            p = self._sigmoid(y_hat)
            g = p - y
            h = p * (1.0 - p)

            tree = XGBTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                reg_lambda=self.reg_lambda,
                gamma=self.gamma,
            )
            tree.fit(X, g, h)

            update = tree.predict(X)
            y_hat = y_hat + self.learning_rate * update
            self.trees.append(tree)

    def predict_logits(self, X):
        logits = torch.full((X.shape[0],), fill_value=self.base_score, dtype=X.dtype, device=X.device)
        for tree in self.trees:
            logits = logits + self.learning_rate * tree.predict(X)
        return logits

    def predict_proba(self, X):
        logits = self.predict_logits(X)
        return torch.sigmoid(logits)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).float()